# 추론·상관·회귀·샘플링·스케일링 보충

추론통계·상관·회귀·샘플링·스케일링을 다루는 보충 단원입니다. 가상 예제의 결론을 실제 사람·서비스에 일반화하지 마세요.

**학습 목표:** 신뢰구간과 검정, 두 집단 비교와 효과 크기, 상관과 회귀, 샘플링과 편향, 스케일링과 데이터 누출

공식 → 직접 구현 또는 계산 → 라이브러리 대조 → 해석 순서로 학습합니다.

## 실행 준비

Python 3.11 이상, NumPy·pandas·SciPy·Matplotlib이 필요합니다. 없는 경우 별도 셀에서 `%pip install numpy pandas scipy matplotlib`을 한 번 실행하세요. 새 커널에서 위에서 아래로 실행합니다. 외부 데이터 다운로드는 없습니다. 그래프는 폰트 호환을 위해 영문 제목을 쓰고 해석은 한국어로 제공합니다.

**표기:** 기본 기술 분산은 ddof=0, 모집단 분산 추정·평균 추론은 ddof=1을 명시합니다. 난수는 seed=42(문제에 지정 시 해당 seed)입니다.

In [1]:
import math
from collections import Counter
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
# 외부 파일 없이 재현 가능한 학습용 가상 자료
x = np.array([12, 15, 14, 10, 18, 20, 22, 25, 30, 28], dtype=float)
y = np.array([2, 4, 6, 8, 10, 12, 14, 40], dtype=float)
plt.rcParams['axes.unicode_minus'] = False


## 1. 신뢰구간과 검정

독립 표본이 정규모집단에서 왔거나 평균 추론의 근사가 적절할 때, 미지 모표준편차에는 t 방법을 사용합니다. 평균의 95% 신뢰구간=x̄±t*.s/√n입니다. 같은 절차를 반복했을 때 구간의 약 95%가 고정된 모평균을 포함한다는 뜻입니다.

In [2]:
se=x.std(ddof=1)/np.sqrt(len(x));critical=stats.t.ppf(.975,len(x)-1)
ci=(x.mean()-critical*se,x.mean()+critical*se)
print(ci,stats.t.interval(.95,df=len(x)-1,loc=x.mean(),scale=se))
t=(x.mean()-18)/se
print(t,2*stats.t.sf(abs(t),len(x)-1),stats.ttest_1samp(x,18))

(np.float64(14.522499109611157), np.float64(24.27750089038884)) (np.float64(14.522499109611157), np.float64(24.27750089038884))
0.6493120348082609 0.5323554284682724 TtestResult(statistic=np.float64(0.6493120348082609), pvalue=np.float64(0.5323554284682724), df=np.int64(9))


**해석 포인트:** 구간은 20을 포함하며 .05에서 기각할 근거가 부족합니다. 이는 μ=20의 입증이 아닙니다. 작은 독립 표본의 정규성·이상치와 표집 방법을 확인합니다. 표준편차와 표준오차를 혼동하지 않습니다.

**직접 확인:** 대응 실습의 문제 1에서 다른 입력으로 적용해 보세요.

## 2. 두 집단 비교와 효과 크기

서로 다른 독립 집단 평균 비교에는 분산을 같다고 강제하지 않는 Welch t검정을 사용할 수 있습니다. 같은 사람의 전후 자료는 대응 t검정이며 독립 검정을 쓰지 않습니다. p값과 함께 평균 차이·신뢰구간·실무 기준을 봅니다. 다중 검정에는 보정이 필요할 수 있습니다.

In [3]:
a=np.array([10,12,11,13,14]);b=np.array([14,15,12,16,17])
print('차이',b.mean()-a.mean(),stats.ttest_ind(b,a,equal_var=False))

차이 2.8000000000000007 TtestResult(statistic=np.float64(2.514474228374849), pvalue=np.float64(0.03716164104091459), df=np.float64(7.711133400200603))


**해석 포인트:** 차이는 약 2.833점입니다. 같은 사람의 차이들이 독립이고 차이의 정규성 가정이 적절해야 합니다. 통계적으로 유의해도 교육 효과의 실무적 크기와 무작위 배정 여부를 별도로 판단합니다.

**직접 확인:** 대응 실습의 문제 2에서 다른 입력으로 적용해 보세요.

## 3. 상관과 회귀

Pearson r은 선형 관계의 방향과 강도입니다. 상관은 인과가 아닙니다. 회귀는 y≈절편+기울기*x로 조건부 평균을 설명·예측합니다. 기울기는 x 한 단위 증가에 따른 y의 변화량입니다. 상수 변수에서는 상관이 정의되지 않습니다. 추론에는 오차 독립성·등분산성 등 추가 가정이 필요합니다.

In [4]:
a=np.array([1,2,3,4,5.]);b=np.array([3,5,4,8,10.])
slope=sum((a-a.mean())*(b-b.mean()))/sum((a-a.mean())**2)
intercept=b.mean()-slope*a.mean()
r=sum((a-a.mean())*(b-b.mean()))/np.sqrt(sum((a-a.mean())**2)*sum((b-b.mean())**2))
print(slope,intercept,r,stats.linregress(a,b))
assert np.isclose(r,np.corrcoef(a,b)[0,1])

1.7 0.9000000000000004 0.9219544457292888 LinregressResult(slope=np.float64(1.7), intercept=np.float64(0.9000000000000004), rvalue=np.float64(0.9219544457292886), pvalue=np.float64(0.02586458407871885), stderr=np.float64(0.41231056256176624), intercept_stderr=np.float64(1.367479433117735))


**해석 포인트:** 양의 선형 관계를 보입니다. 다른 학습 배경 등의 교란이 있어 상관만으로 인과를 주장할 수 없습니다. 관측 범위를 벗어난 장시간 예측은 직선 관계를 보장하지 않습니다.

**직접 확인:** 대응 실습의 문제 3에서 다른 입력으로 적용해 보세요.

## 4. 샘플링과 편향

단순무작위는 모든 개체에 같은 선택 기회를 줍니다. 층화 추출은 집단별 표집을 보장합니다. 편의 추출의 선택 편향은 표본 수 증가만으로 해결되지 않습니다. 비복원 추출은 중복이 없고, 부트스트랩은 복원 추출을 반복합니다.

In [5]:
pop=pd.DataFrame({'id':range(100),'group':['A']*80+['B']*20})
print(pop.sample(n=10,random_state=42))
strat=pd.concat([g.sample(n=k,random_state=42) for (_,g),k in zip(pop.groupby('group'),[8,2])])
print(strat.groupby('group').size())

    id group
83  83     B
53  53     A
70  70     A
45  45     A
44  44     A
39  39     A
22  22     A
80  80     B
10  10     A
0    0     A
group
A    8
B    2
dtype: int64


**해석 포인트:** A 6명, B 4명으로 모집단 비율을 보존합니다. 각 층 안에서도 무작위 추출이 필요합니다. 이 설계는 집단 누락을 줄이지만 모집단 명부의 누락 편향은 해결하지 못합니다.

**직접 확인:** 대응 실습의 문제 4에서 다른 입력으로 적용해 보세요.

## 5. 스케일링과 데이터 누출

Z 표준화는 (x−평균)/표준편차, Min-Max는 (x−최소)/(최대−최소)입니다. 정규성을 만들지 않습니다. 학습 데이터에서만 평균·최소·최대를 구하고 검증·테스트에 그대로 적용해야 데이터 누출을 피합니다. Min-Max의 새 값은 [0,1]을 벗어날 수 있습니다.

In [6]:
train=np.array([10,20,30.]);test=np.array([40.])
mu,sigma=train.mean(),train.std(ddof=0)
print((train-mu)/sigma,(test-mu)/sigma)
print((test-train.min())/np.ptp(train))

[-1.22474487  0.          1.22474487] [2.44948974]
[1.5]


**해석 포인트:** Min-Max 테스트 결과는 약 1.333과 1.667입니다. test를 이용해 범위를 다시 맞추면 누출이 생깁니다. 상수 열은 분모 0이므로 제거하거나 명시된 정책으로 0을 부여하는 등 별도 처리합니다.

**직접 확인:** 대응 실습의 문제 5에서 다른 입력으로 적용해 보세요.

## 복습 순서

1. 관측 대상과 통계량의 의미를 말로 설명합니다.
2. 분모·단위·독립성·분포 가정을 확인합니다.
3. 직접 계산과 라이브러리 결과를 비교합니다.
4. 코드 결과와 실제로 주장할 수 있는 범위를 구분합니다.

**학습 이동:** `Statistics_4_inference_extension.ipynb` → `Statistics_4_inference_extension_Practice.ipynb` → `Statistics_4_inference_extension_Answer.ipynb`.